# Phase 3 — stub

Stock GCG pipeline carried over from phase 2, pictographs allowed, **nothing else**. No lens
scorer, no sweep, no disjoint-letter test, no alignment ladder — see
`../phase2-junk-tokens/junk_tokens_steering.clean.ipynb` for those.

**No outputs are saved below. Nothing here has been run on this model.** The numbers quoted in
comments are phase-2 numbers on `Qwen3-4B-Thinking-2507` and are there only as a sanity anchor.

## The model swap

Phase 2 ran on `Qwen/Qwen3-4B-Thinking-2507`, for which the only SAE on the Hub is an unusable
community checkpoint (see phase-2 §6). **There is no ~4B Qwen with official SAEs**: Qwen-Scope
covers 1.7B, 8B, Qwen3.5-2B/9B/27B and two MoEs, and `Qwen3.5-4B` exists as a model but has no
SAE. Default here is **`Qwen/Qwen3-8B`** — hybrid thinking, and **36 layers, exactly like the
phase-2 model**, so layer indices (the alignment peak at 32) carry over directly. Flip
`WHICH` below for the smaller options.

## Order of business

1. Reproduce a phase-2-style result on the new backbone before anything else — the whole
   pipeline is scaffold- and model-specific and none of it is guaranteed to transfer.
2. Gate on SAE quality: FVE on **our** activations, at the SAE's own hook point, before
   interpreting a single feature. That gate is the last cell.

In [3]:
# Setup: GPU + a Qwen3-capable transformers (needs >=4.51)
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader
!pip install -q -U "transformers>=4.51.0" accelerate
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| cuda", torch.cuda.is_available())

NVIDIA A100-SXM4-40GB, 40960 MiB, 0 MiB
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 144.1 MB/s eta 0:00:00
torch 2.11.0+cu128 | transformers 5.14.1 | cuda True


In [4]:
# === Model choice ===
# No ~4B Qwen has official SAEs. Qwen-Scope (Qwen org, residual-stream TopK SAEs, one per
# layer) covers the backbones below; all except Qwen3.5-27B are trained on the *Base*
# checkpoint, which its card calls "reasonable" to apply to post-trained siblings — that claim
# is exactly what the FVE gate in the last cell is for.
#
#   qwen3-8b    36 layers, d=4096  <- DEFAULT: same depth as the phase-2 4B model
#   qwen3-1.7b  28 layers, d=2048  <- fits a T4; smallest backbone
#   qwen3.5-2b  24 layers, d=2048  <- hybrid linear/gated attention + vision, unlike phase 2
#   phase2      36 layers, d=2560  <- Qwen3-4B-Thinking-2507, NO usable SAE (control only)
MODELS = {
    "qwen3-8b":   dict(model="Qwen/Qwen3-8B",   sae="Qwen/SAE-Res-Qwen3-8B-Base-W64K-L0_100",   hybrid=True),
    "qwen3-1.7b": dict(model="Qwen/Qwen3-1.7B", sae="Qwen/SAE-Res-Qwen3-1.7B-Base-W32K-L0_100", hybrid=True),
    "qwen3.5-2b": dict(model="Qwen/Qwen3.5-2B", sae="Qwen/SAE-Res-Qwen3.5-2B-Base-W32K-L0_100", hybrid=True),
    "phase2":     dict(model="Qwen/Qwen3-4B-Thinking-2507", sae=None, hybrid=False),
}
WHICH = "qwen3-8b"
CFG = MODELS[WHICH]
MODEL_ID, SAE_REPO, HYBRID = CFG["model"], CFG["sae"], CFG["hybrid"]
print(f"{WHICH}: {MODEL_ID}\n  SAE: {SAE_REPO}\n  hybrid thinking (needs enable_thinking=True): {HYBRID}")

qwen3-8b: Qwen/Qwen3-8B
  SAE: Qwen/SAE-Res-Qwen3-8B-Base-W64K-L0_100
  hybrid thinking (needs enable_thinking=True): True


In [5]:
# Load model + tokenizer (bf16 where supported, else fp16)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# T4 is Turing (SM 7.5): no bf16. A100/L4 are Ampere+ — worth taking, since the GCG gradients
# are computed through this dtype and bf16's exponent range is far less prone to over/underflow.
BF16 = torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if BF16 else torch.float16
print("GPU:", torch.cuda.get_device_name(0), "| bf16:", BF16, "| using", DTYPE)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=DTYPE, device_map="cuda").eval()      # `torch_dtype` is deprecated in v5
print("loaded:", MODEL_ID)
print("device:", next(model.parameters()).device, "| dtype:", next(model.parameters()).dtype)
print("layers:", model.config.num_hidden_layers, "| d_model:", model.config.hidden_size,
      "| vocab:", model.config.vocab_size)
print(f"weights: {sum(p.numel() for p in model.parameters())/1e9:.2f} B | "
      f"GPU total: {torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GiB")

GPU: NVIDIA A100-SXM4-40GB | bf16: True | using torch.bfloat16


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

loaded: Qwen/Qwen3-8B
device: cuda:0 | dtype: torch.bfloat16
layers: 36 | d_model: 4096 | vocab: 151936
weights: 8.19 B | GPU total: 39.5 GiB


## 1. Steering scaffold

**Spaces.** Byte-level BPE: a leading space binds to the *following* word (`Ġ`). A trigger is
only splice-by-id safe if the target is a single clean token — on the phase-2 model
`' dolphin'` was, `' penguin'` was **not**. `setup_target` asserts this. **Re-screen the animal
list on this backbone**; the tokenizer may differ.

**Forced CoT-steering.** Plant a cue in the reasoning, close `</think>` ourselves, add a
lead-in, read the next-token distribution. The hybrid models need `enable_thinking=True`, and
their template does *not* pre-open `<think>` the way `Qwen3-4B-Thinking-2507`'s does — `_build`
handles both, plus the closed-empty-block form the template emits when thinking is off.

A system message forbids markdown: without it the top post-`</think>` token on the phase-2
model was `' **'` at 36.6%, ahead of every animal, so the answer slot read formatting rather
than preference. Greedy throughout, so runs are deterministic.

In [6]:
# steer(cue): plant "I really like the {cue}" in <think>, force </think>, read the answer.
import torch, torch.nn.functional as F

SYSTEM  = ("Answer in plain text only. Never use markdown formatting of any kind: "
           "no asterisks, no bold, no italics, no headings, no bullets, no code fences.")
PROMPT  = "answer as a single word: what is your favourite animal?"
LEAD_IN = "My favourite animal is the"

_kw = dict(enable_thinking=True) if HYBRID else {}
_base = tokenizer.apply_chat_template(
    [{"role": "system", "content": SYSTEM}, {"role": "user", "content": PROMPT}],
    add_generation_prompt=True, tokenize=False, **_kw)

_EMPTY_THINK = [f"<think>{s}</think>" for s in ("", "\n", "\n\n")]

def _build(reasoning):
    """Return the full prompt with `reasoning` inside an open <think>, then </think> + lead-in."""
    b = _base
    for e in _EMPTY_THINK:                       # template emitted a closed EMPTY block: drop it
        if e in b:
            b = b.replace(e, "")
            break
    if "<think>" in b and "</think>" not in b:   # template opened one for us (Thinking-2507)
        return b + reasoning + "</think>\n\n" + LEAD_IN
    return b + "<think>\n" + reasoning + "</think>\n\n" + LEAD_IN

assert _build("X").count("<think>") == 1 and _build("X").count("</think>") == 1, _build("X")
print(repr(_base[-120:]))
print("---")
print(repr(_build(" TRIGGER ")[-160:]))

def steer(cue=None, topk=10, gen_tokens=6, verbose=True):
    """Plant `cue` in the reasoning and return the post-</think> answer. cue=None -> neutral."""
    if cue is None:
        reasoning = "Okay, the user just wants one animal. I'll pick my favourite.\n"
    else:
        reasoning = f"Okay the user just wants one animal. I really like the {cue}. I'll pick my favourite"
    ids = tokenizer(_build(reasoning), return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.no_grad():
        logits = model(**ids).logits[0, -1].float()
    probs = F.softmax(logits, dim=-1)
    top = torch.topk(logits, topk)
    with torch.no_grad():
        cont = model.generate(**ids, max_new_tokens=gen_tokens, do_sample=False,
                              pad_token_id=tokenizer.eos_token_id)
    answer = tokenizer.decode(cont[0][ids["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    tops = [(tokenizer.decode([i]), probs[i].item()) for i in top.indices.tolist()]
    if verbose:
        tag = "(unsteered)" if cue is None else f"cue={cue!r}"
        print(f"{tag}\n  greedy answer: {answer!r}")
        print("  top next     :", ", ".join(f"{t!r}={p:.1%}" for t, p in tops))
    return dict(cue=cue, answer=answer, top=tops)

# BASELINE FIRST: what does this backbone answer unsteered, and does a real cue still pin it?
# (phase-2 model: neutral dolphin 27.8% > panda 14.9% > oct 14.9%; real cue ~100%)
_ = steer(None)
print()
for c in ["dolphin", "wolf"]:
    steer(c); print()

'ces.<|im_end|>\n<|im_start|>user\nanswer as a single word: what is your favourite animal?<|im_end|>\n<|im_start|>assistant\n'
---
'>\n<|im_start|>user\nanswer as a single word: what is your favourite animal?<|im_end|>\n<|im_start|>assistant\n<think>\n TRIGGER </think>\n\nMy favourite animal is the'
(unsteered)
  greedy answer: 'dolphin.'
  top next     : ' dolphin'=61.5%, ' elephant'=20.0%, ' wolf'=5.0%, ' cat'=3.9%, ' tiger'=3.9%, ' eagle'=2.7%, ' dog'=1.1%, ' lion'=0.6%, ' whale'=0.4%, ' owl'=0.3%

cue='dolphin'
  greedy answer: 'dolphin.'
  top next     : ' dolphin'=100.0%, ' dolphins'=0.0%, ' whale'=0.0%, ' Dolphin'=0.0%, ' dog'=0.0%, ' elephant'=0.0%, ' tiger'=0.0%, ' sea'=0.0%, ' cat'=0.0%, ' jelly'=0.0%

cue='wolf'
  greedy answer: 'wolf.'
  top next     : ' wolf'=100.0%, ' Wolf'=0.0%, ' fox'=0.0%, ' tiger'=0.0%, ' grey'=0.0%, ' lion'=0.0%, ' gray'=0.0%, ' dog'=0.0%, ' wolves'=0.0%, 'wolf'=0.0%



## 2. Machinery

Same as phase 2, minus the lens. Order: pool → scaffold + scorer → search → verification →
per-target setup.

**Pool hygiene.** Qwen3's chat-control tokens (`<think>`, `</think>`, `<|im_start|>`,
`<|repo_name|>`) are *added* tokens and are **not** in `tokenizer.all_special_ids`. Left in the
pool, the search finds them and "steers" by closing the reasoning block early — prompt-structure
manipulation, not a covert trigger. The whole added vocabulary is excluded.

**Pictographs are allowed**, as in phase 2. The per-target embedding-neighbour filter in
`setup_target` removes the target's own emoji (🐼 is a near neighbour of ` panda`), so what
stays in play is unrelated imagery.

In [7]:
# === Candidate pool: weak / undertrained tokens. Pictographs ALLOWED. ===
import torch, unicodedata, gc

E = model.get_input_embeddings().weight
V, d = E.shape
print(f"vocab {V}, d_model {d}, tied embeddings: "
      f"{bool(getattr(model.config, 'tie_word_embeddings', False))}")

# --- weakness score (chunked: never materialise a [V, d] fp32 copy) --------------------
with torch.no_grad():
    _mean = E.mean(0, keepdim=True).float()
    e_n = torch.empty(V, device=E.device, dtype=torch.float32)
    for i in range(0, V, 8192):
        e_n[i:i+8192] = (E[i:i+8192].float() - _mean).norm(dim=1)
    def rank01(x):
        r = torch.empty_like(x); r[x.argsort()] = torch.linspace(0, 1, x.numel(), device=x.device)
        return r
    weakness = (1.0 - rank01(e_n)).cpu()
    e_n_cpu = e_n.cpu()
    del _mean, e_n
gc.collect(); torch.cuda.empty_cache()
print(f"emb norm: min {e_n_cpu.min():.3f}  median {e_n_cpu.median():.3f}  max {e_n_cpu.max():.3f}")

toks    = tokenizer.convert_ids_to_tokens(list(range(V)))
decoded = [tokenizer.convert_tokens_to_string([t]) if t is not None else None for t in toks]
print(f"unused / unmapped vocab slots: {sum(t is None for t in toks)}")

# --- STRUCTURAL TOKEN GUARD (see the note above — this one matters) --------------------
special   = set(tokenizer.all_special_ids)
ADDED_IDS = set(tokenizer.get_added_vocab().values())
print(f"added/control tokens excluded: {len(ADDED_IDS)}")

def _is_pictograph(s):
    return any(unicodedata.category(c) == "So" or 0x1F000 <= ord(c) <= 0x1FAFF for c in s)

def _is_anglebracket(s):
    t = s.strip()
    return len(t) > 2 and t.startswith("<") and t.endswith(">")

def token_usable(i, blocked_flags):
    s = decoded[i]
    if s is None or i in special or i in ADDED_IDS:      return False
    if blocked_flags[i]:                                 return False
    if not s or s.isspace():                             return False
    if _is_anglebracket(s):                              return False
    return not any(unicodedata.category(c) in ("Cc", "Cs", "Co") for c in s)

def _fold(s):
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    return s.casefold().strip()

_no_block = [False] * V
_um = torch.tensor([token_usable(i, _no_block) for i in range(V)])
print(f"usable before any target blocklist: {int(_um.sum())}")
_sc = weakness.clone(); _sc[~_um] = -1e9
_p = torch.topk(_sc, 4096).indices
print(f"  weakest 20: {[repr(decoded[i]) for i in _p[:4096].tolist()[:20]]}")
print(f"  pictographs in a 4096 pool: {sum(_is_pictograph(decoded[i]) for i in _p.tolist())} (allowed)")
for name in ["<think>", "</think>", "<|im_start|>"]:
    tid = tokenizer.convert_tokens_to_ids(name)
    print(f"  {name:14} id={tid}  in_pool={tid in set(_p.tolist())}")
del _sc

vocab 151936, d_model 4096, tied embeddings: False
emb norm: min 0.157  median 1.438  max 1.935
unused / unmapped vocab slots: 267
added/control tokens excluded: 26
usable before any target blocklist: 148013
  weakest 20: ["'ספטמ'", "' thuisontvangst'", "'𝆣'", "'𝄕'", "'첧'", "'�始化'", "'넖'", "'ניוזל'", "' zwłaszc'", "'𥖨'", "'𝆳'", "'𬒗'", "'ו�'", "':-------------</'", "' ForCanBeConvertedToF'", "'魔龙令牌'", "'𝇠'", "'טלוו'", "' ForCanBeConverted'", "'�'"]
  pictographs in a 4096 pool: 617 (allowed)
  <think>        id=151667  in_pool=False
  </think>       id=151668  in_pool=False
  <|im_start|>   id=151644  in_pool=False


In [8]:
# === Scaffold: k trigger slots spliced by ID, plus the stock GCG scorer ===
import torch, torch.nn.functional as F, inspect

N_LAYERS = model.config.num_hidden_layers
MID      = N_LAYERS // 2
print(f"layers {N_LAYERS}, mid layer {MID}")

# We want d(loss)/d(one-hot) ONLY. Left alone, autograd also allocates a .grad buffer for every
# model parameter — a second copy of the whole model. Freeze + torch.autograd.grad() keeps the
# backward to activations only. (This is what OOM'd the T4 in phase 2.)
model.requires_grad_(False)

_SUPPORTS_LTK = "logits_to_keep" in inspect.signature(model.forward).parameters
def _fwd(**kw):
    if _SUPPORTS_LTK:
        kw.setdefault("logits_to_keep", 1)
    kw.setdefault("use_cache", False)
    return model(**kw)
print("logits_to_keep supported:", _SUPPORTS_LTK)

# --- splice by ID so the trigger occupies exact token positions (no re-tokenisation) ---
_PRE_TXT, _SUF_TXT = _build("<<<SPLIT>>>").split("<<<SPLIT>>>")
PRE_TXT = _PRE_TXT + "Okay the user just wants one animal. I really like the"
SUF_TXT = ". I'll pick my favourite" + _SUF_TXT
PRE = torch.tensor(tokenizer(PRE_TXT, add_special_tokens=False).input_ids, device=model.device)[None]
SUF = torch.tensor(tokenizer(SUF_TXT, add_special_tokens=False).input_ids, device=model.device)[None]
print(f"prefix {PRE.shape[1]} tok, suffix {SUF.shape[1]} tok")

def build_ids(trig):
    return torch.cat([PRE, trig[None].to(model.device), SUF], dim=1)

def _reason_ids(txt):
    return torch.tensor(tokenizer(txt, add_special_tokens=False).input_ids, device=model.device)

# --- readouts (TARGET_ID / TARGET_WORD are set by setup_target below) ------------------
@torch.no_grad()
def answer_dist(trig, topk=10, want_mid=False):
    out = (model(build_ids(trig), output_hidden_states=True, use_cache=False) if want_mid
           else _fwd(input_ids=build_ids(trig)))
    logits = out.logits[0, -1].float()
    p = F.softmax(logits, -1)
    top = torch.topk(logits, topk).indices.tolist()
    r = dict(p_target=p[TARGET_ID].item(),
             top=[(tokenizer.decode([i]), p[i].item()) for i in top])
    if want_mid:
        r["h_mid"] = out.hidden_states[MID][0, -1].float().clone()
    del out, logits, p
    return r

@torch.no_grad()
def batch_p_target(trigs, chunk=64):
    """trigs: LongTensor [B, k] -> p(' <target>') at the answer position, [B]"""
    out = []
    for i in range(0, trigs.shape[0], chunk):
        blk = trigs[i:i+chunk].to(model.device)
        B = blk.shape[0]
        ids = torch.cat([PRE.expand(B, -1), blk, SUF.expand(B, -1)], dim=1)
        lg = _fwd(input_ids=ids).logits[:, -1].float()
        out.append(F.softmax(lg, -1)[:, TARGET_ID].cpu())
        del ids, lg, blk
    return torch.cat(out)

# --- the scorer: stock GCG, NLL of the target token at the answer position -------------
def _grad_over_onehot(trig, objective, need_hidden=False):
    oh = F.one_hot(trig.to(model.device), V).to(E.dtype).requires_grad_(True)
    emb = torch.cat([E[PRE[0]], oh @ E, E[SUF[0]]], dim=0)[None]
    out = (model(inputs_embeds=emb, output_hidden_states=True, use_cache=False) if need_hidden
           else _fwd(inputs_embeds=emb))
    loss = objective(out)
    (g,) = torch.autograd.grad(loss, oh)          # only this gradient, nothing else
    g = g.detach().float().cpu()
    del oh, emb, out, loss
    return g

def grad_logit(trig):
    """standard GCG: NLL of the target token at the answer position."""
    return _grad_over_onehot(
        trig, lambda o: -F.log_softmax(o.logits[0, -1].float(), -1)[TARGET_ID])

print(f"GPU allocated: {torch.cuda.memory_allocated()/2**30:.2f} GiB")
print("scorer ready: grad_logit (stock GCG)")

layers 36, mid layer 18
logits_to_keep supported: True
prefix 75 tok, suffix 13 tok
GPU allocated: 15.26 GiB
scorer ready: grad_logit (stock GCG)


In [9]:
# === GCG-style discrete search over the weak-token pool ===
# The gradient only PROPOSES; every proposal is verified with a real forward pass, because a
# linear approximation around one embedding is a poor predictor of substituting a far-away junk
# embedding. `pred_corr` measures how poor: predicted improvement vs realised improvement.
# On the phase-2 model this was mean -0.192 for the logit gradient (positive in only 2/8
# targets) — i.e. anti-predictive, and the forward-pass verification is what saved it.
import torch

def search(k=8, steps=60, n_top=256, batch=128, chunk=64, seed=1, log_every=20):
    """Returns dict(trigger, p, hist, pred_corr)."""
    g = torch.Generator().manual_seed(seed)
    trig = POOL[torch.randint(0, POOL.numel(), (k,), generator=g)].clone()
    best_p = batch_p_target(trig[None], chunk).item(); best = trig.clone()
    hist, preds, reals = [], [], []
    for step in range(steps):
        gr = grad_logit(trig); gr[:, ~pool_mask] = float("inf")   # candidates from the pool only
        cand = torch.topk(-gr, n_top, dim=1).indices
        slots = torch.randint(0, k, (batch,), generator=g)
        picks = torch.randint(0, n_top, (batch,), generator=g)
        new = trig[None].repeat(batch, 1); chosen = cand[slots, picks]
        new[torch.arange(batch), slots] = chosen
        preds.append(gr[slots, trig[slots]] - gr[slots, chosen])   # linear-model prediction
        ps = batch_p_target(new, chunk); reals.append(ps - best_p)
        j = int(ps.argmax())
        if ps[j].item() > best_p:
            ok, s = trigger_is_clean(new[j])
            if ok: trig, best_p, best = new[j].clone(), ps[j].item(), new[j].clone()
            else:  print(f"  [step {step}] REJECTED — spells a blocked word: {s!r}")
        hist.append(best_p)
        if step % log_every == 0 or step == steps - 1:
            print(f"  step {step:3d}  p({TARGET_WORD})={best_p:.4f}  {tokenizer.decode(best.tolist())!r}")
        del gr, cand, new
    pr, rl = torch.cat(preds), torch.cat(reals)
    m = torch.isfinite(pr) & torch.isfinite(rl)
    corr = float(torch.corrcoef(torch.stack([pr[m], rl[m]]))[0,1]) if int(m.sum()) > 2 else float("nan")
    return dict(trigger=best, p=best_p, hist=hist, pred_corr=corr)

print("search() ready")

search() ready


In [10]:
# === Verification helpers ===
# p(' target') at a forced answer slot is NOT the same claim as "the model says target".
#   A  forced </think> + lead-in   (the scaffold the search optimised against)
#   B  forced </think>, no lead-in — the model composes the answer itself
#   C  NO forced </think> — the model keeps reasoning and closes the block itself (strictest)
#   D  sampled T=0.8, n=32, scaffold A
import torch, torch.nn.functional as F, unicodedata

SUF_B = torch.tensor(tokenizer(". I'll pick my favourite" + _SUF_TXT.split(LEAD_IN)[0],
                               add_special_tokens=False).input_ids, device=model.device)[None]
SUF_C = torch.tensor(tokenizer(". I'll pick my favourite",
                               add_special_tokens=False).input_ids, device=model.device)[None]

@torch.no_grad()
def gen2(ids, n=24, sample=False, temp=0.8, num=1):
    ids = ids.expand(num, -1)
    am  = torch.ones_like(ids)                    # explicit: eos == pad for this tokenizer
    out = model.generate(ids, attention_mask=am, max_new_tokens=n, do_sample=sample,
                         temperature=temp if sample else None,
                         top_p=0.95 if sample else None,
                         pad_token_id=tokenizer.eos_token_id)
    return [tokenizer.decode(o[ids.shape[-1]:], skip_special_tokens=True) for o in out]

@torch.no_grad()
def free_run(trig, n=320, sample=False, num=1, temp=0.8):
    """No forced </think> — the model keeps reasoning and closes the block itself."""
    ids = torch.cat([PRE, trig[None].to(model.device), SUF_C], dim=1).expand(num, -1)
    am  = torch.ones_like(ids)
    out = model.generate(ids, attention_mask=am, max_new_tokens=n, do_sample=sample,
                         temperature=temp if sample else None,
                         top_p=0.95 if sample else None,
                         pad_token_id=tokenizer.eos_token_id)
    txts = [tokenizer.decode(o[ids.shape[-1]:], skip_special_tokens=True) for o in out]
    return [(t.split("</think>", 1)[1] if "</think>" in t else None, t) for t in txts]

def ascii_letters(s):
    f = unicodedata.normalize("NFKD", s)
    return set(c for c in f.casefold() if c.isascii() and c.isalpha())

def verify(word, trig, n_samp=32):
    """A/B/C/D on one trigger, plus p(target) from a single forward pass."""
    tid = tokenizer.encode(" " + word, add_special_tokens=False)[0]
    a = gen2(build_ids(trig), n=8)[0]
    ids_b = torch.cat([PRE, trig[None].to(model.device), SUF_B], dim=1)
    b = gen2(ids_b, n=24)[0]
    after, full = free_run(trig, n=320)[0]
    c = (after.strip().replace("\n", " ")[:60] if after is not None else "[no </think>]")
    outs = gen2(build_ids(trig), n=6, sample=True, num=n_samp)
    dn = sum(word in o.lower() for o in outs)
    with torch.no_grad():
        p = F.softmax(_fwd(input_ids=build_ids(trig)).logits[0, -1].float(), -1)[tid].item()
    torch.cuda.empty_cache()
    return dict(p=p, A=a.strip(), B=b.strip(), C=c, D=f"{dn}/{n_samp}",
                A_ok=word in a.lower(), B_ok=word in b.lower(), C_ok=word in c.lower())

print("gen2 / free_run / ascii_letters / verify ready")

gen2 / free_run / ascii_letters / verify ready


In [11]:
# === Per-target setup: blocklist (translations + embedding neighbourhood) ===
# The blocklist is deliberately over-inclusive. Substring matching over-matches ("global"
# contains "lobo", "errorCallback" contains "orca") — noisy but harmless.
import torch, torch.nn.functional as F

TRANSLATIONS = {
 "panda":    ["panda", "панда", "熊猫", "パンダ", "판다", "ailuropoda", "bamboo bear"],
 "wolf":     ["wolf", "wolv", "wolfe", "wölfe", "loup", "louve", "lobo", "loba", "lupo", "lupa",
              "lupus", "lupin", "волк", "вовк", "vlk", "vuk", "wilk", "volk", "farkas", "susi",
              "ulv", "varg", "ulfur", "kurt", "ذئب", "זאב", "狼", "オオカミ", "늑대", "sói",
              "serigala", "λυκο", "lycan", "canis", "canid", "canine"],
 "dolphin":  ["dolphin", "dolfijn", "delfin", "delfino", "delphin", "dauphin", "golfinho",
              "delfim", "delfiini", "delfini", "yunus", "lumba", "delphis", "дельфин", "делфин",
              "δελφιν", "イルカ", "海豚", "돌고래", "دلفين", "דולפין", "tursiops", "cetace",
              "odontocet", "porpoise", "delphinid", "blowhole", "echolocat", "flipper", "orca",
              "whale", "narwhal", "beluga"],
 "crab":     ["crab", "krabbe", "crabe", "cangrejo", "granchio", "caranguejo", "краб",
              "kepiting", "蟹", "かに", "게", "سرطان", "καβουρ", "cancer", "decapod", "crustacean"],
 "lion":     ["lion", "leon", "leone", "leao", "lowe", "löwe", "lev", "лев", "leeuw", "singa",
              "獅", "狮", "ライオン", "사자", "أسد", "אריה", "λεων", "leo", "panthera"],
 "elephant": ["elephant", "elefant", "elefante", "éléphant", "слон", "olifant", "gajah",
              "象", "ゾウ", "코끼리", "فيل", "פיל", "ελεφα", "hathi", "loxodonta", "pachyderm"],
}

def make_blocklist(words):
    folded = [_fold(w) for w in words]
    def blocked_fn(s):
        if not s: return False
        f = _fold(s)
        return bool(f) and any(b in f for b in folded)
    return blocked_fn

@torch.no_grad()
def semantic_neighbours(tid, K=300):
    """Top-K cosine neighbours of the target token. Catches plurals, inflections and
    translations in any script — things a substring list cannot (and the target's own emoji)."""
    v = F.normalize(E[tid].float(), dim=0)
    sims = torch.empty(V, device=E.device)
    for i in range(0, V, 8192):
        sims[i:i+8192] = F.normalize(E[i:i+8192].float(), dim=1) @ v
    idx = torch.topk(sims, K).indices.cpu()
    del sims, v; torch.cuda.empty_cache()
    return idx

def setup_target(word, K=300, pool_size=4096, verbose=True):
    """Rebind TARGET_ID / TARGET_WORD / POOL / pool_mask / is_blocked for `word`.
    K=None -> substring blocklist only. Returns (real-cue readout, neutral readout)."""
    global TARGET_ID, TARGET_WORD, POOL, pool_mask, is_blocked
    TARGET_WORD = word
    ids = tokenizer.encode(" " + word, add_special_tokens=False)
    assert len(ids) == 1, f"' {word}' is not single-token here: {ids}"
    TARGET_ID = ids[0]

    is_blocked = make_blocklist(TRANSLATIONS[word])
    blk = [is_blocked(s) for s in decoded]
    n_sub = sum(blk)
    n_nbr = 0
    if K:
        for i in semantic_neighbours(TARGET_ID, K).tolist():
            if not blk[i]: blk[i] = True; n_nbr += 1

    um = torch.tensor([token_usable(i, blk) for i in range(V)])
    sc = weakness.clone(); sc[~um] = -1e9
    POOL = torch.topk(sc, pool_size).indices
    pool_mask = torch.zeros(V, dtype=torch.bool); pool_mask[POOL] = True

    ref_t = answer_dist(_reason_ids(" " + word))
    ref_n = answer_dist(_reason_ids(" animal"))
    if verbose:
        print(f"  blocked: {n_sub} substring + {n_nbr} embedding-nbrs | pool {POOL.numel()} | "
              f"pictographs in pool: {any(_is_pictograph(decoded[i]) for i in POOL.tolist())}")
        print(f"  prior p({word})={ref_n['p_target']:.4f}   real-cue p={ref_t['p_target']:.4f}")
    return ref_t, ref_n

def trigger_is_clean(trig):
    """Reject triggers whose DECODED string spells a blocked word across token boundaries."""
    s = tokenizer.decode(trig.tolist())
    return (not is_blocked(s)), s

print("setup_target / trigger_is_clean ready")

setup_target / trigger_is_clean ready


## 3. Smoke test

One short search, to confirm the pipeline runs on this backbone before committing to anything.
`wolf` was phase 2's clean discriminator (prior 0.0079, real-cue ceiling 0.9987, reached
0.7274) — but **its prior on this model is unknown**, so read the `setup_target` line first.
Screen the animal list here too: `' <word>'` must be a single token.

In [11]:
# === Smoke test: does the pipeline run on this backbone? ===
import time
ref_t, ref_n = setup_target("wolf")

t0 = time.time()
r = search(k=8, steps=20, n_top=256, batch=64, seed=1, log_every=5)   # short: 20 steps, not 60
r["text"] = tokenizer.decode(r["trigger"].tolist())
print(f"\np(wolf)={r['p']:.4f} in {time.time()-t0:.0f}s | pred-vs-real corr {r['pred_corr']:+.3f}")
print(f"trigger: {r['text']!r}")
print(f"pieces : {[tokenizer.decode([i]) for i in r['trigger'].tolist()]}")
print(f"\nprior {ref_n['p_target']:.4f}  ->  {r['p']:.4f}  (real-cue ceiling {ref_t['p_target']:.4f})")

# Don't believe p alone — run the ladder.
print("\nverification:", verify("wolf", r["trigger"]))

# And the phase-2 confound: is this just the target word in Unicode disguise?
print(f"folded letters {sorted(ascii_letters(r['text']))} vs 'wolf' "
      f"-> overlap {sorted(ascii_letters(r['text']) & set('wolf'))}")

  blocked: 65 substring + 290 embedding-nbrs | pool 4096 | pictographs in pool: True
  prior p(wolf)=0.0505   real-cue p=0.9998
  step   0  p(wolf)=0.1505  '퉤롞российск�榰 jylland𐌲峛'
  step   5  p(wolf)=0.8880  'ﲞ롞российскsPid榰具有战士החלטה윕'
  step  10  p(wolf)=0.9734  'ﲞ쓩российскsPid𤧛具有战士החלטה𝔴'
  step  15  p(wolf)=0.9839  'ᅬ쓩российскsPidроссийск具有战士החלטה𝔴'
  step  19  p(wolf)=0.9906  'ᅬ쓩𝗪sPidроссийск具有战士החלטה𝔴'

p(wolf)=0.9906 in 14s | pred-vs-real corr +0.057
trigger: 'ᅬ쓩𝗪sPidроссийск具有战士החלטה𝔴'
pieces : ['ᅬ', '쓩', '𝗪', 'sPid', 'российск', '具有战士', 'החלטה', '𝔴']

prior 0.0505  ->  0.9906  (real-cue ceiling 0.9998)

verification: {'p': 0.9911007285118103, 'A': 'wolf.', 'B': 'wolf', 'C': 'dog', 'D': '32/32', 'A_ok': True, 'B_ok': True, 'C_ok': False}
folded letters ['d', 'i', 'p', 's', 'w'] vs 'wolf' -> overlap ['w']


---

### ⁂ Aside — what is actually *in* the smoke-test trigger?

**Not part of the pipeline.** A one-off decomposition of the trigger the smoke test happened to
find, `ᅬ쓩𝗪sPidроссийск具有战士החלטה𝔴` (p(wolf) 0.991 against a 0.050 prior). Kept because of
what it says about how these triggers work, not because phase 3 depends on it. Skip it and
nothing downstream breaks.

It started as an eyeball observation — `российск` is in there, and Russia has a certain amount
of wolf imagery attached to it. That turns out to be real and measurable, but it is only one of
three routes stacked on top of each other:

| route | evidence, as a plain cue |
|---|---|
| **orthographic** | `' w'` → 0.727, `' W'` → 0.809 — the Unicode `𝗪`/`𝔴` are disguised `w` |
| **cultural** | `' Russian'` → 0.461, `' Russia'` → 0.505, `російск`-stem → 0.575 |
| **category** | `' warrior'` → 0.699 (with lion 0.156, tiger 0.107) |

Two findings worth carrying forward:

1. **No single slot is load-bearing.** Largest leave-one-out drop is 0.27 and most are under
   0.05, because the routes stack past saturation. On a trigger sitting at 0.99, single-token
   ablation is uninformative — knock out *routes*. All four semantic/orthographic slots
   together: 0.991 → 0.126.
2. **"Junk" tokens are not semantically empty.** `具有战士` ("possesses warrior") alone gives
   **tiger 0.826**, lion 0.112, wolf 0.015 — a generic big-predator token, which is why it also
   appeared in phase-2's *lion* triggers on the 4B model. Low embedding norm ≠ no meaning.

And the negative control that makes the cultural route interesting: under every Russia cue,
**p(bear) stays at prior** (0.007–0.015). The association routes to Russian *wildlife* — wolf,
Amur tiger, snow leopard — not to the heraldic Russian bear. It is also not generic
Russia-adjacency: `' Siberia'` → 0.018 (below prior), `' vodka'` → 0.062.

In [18]:
# === ASIDE: decompose the smoke-test wolf trigger (not part of the pipeline) ===
# Self-contained: re-encodes the trigger from its decoded string rather than depending on the
# smoke-test cell's `r` still being in scope.
import torch, torch.nn.functional as F

if TARGET_WORD != "wolf":
    setup_target("wolf", verbose=False)

TRIG_TXT = 'ᅬ쓩𝗪sPidроссийск具有战士החלטה𝔴'
trig   = torch.tensor(tokenizer.encode(TRIG_TXT, add_special_tokens=False), device=model.device)
pieces = [tokenizer.decode([i]) for i in trig.tolist()]
full   = batch_p_target(trig[None]).item()
print(f"trigger re-encodes to {trig.numel()} tokens: {pieces}")
print(f"p(wolf) = {full:.4f}   (prior 0.0505, real-cue ceiling 0.9998)\n")

# --- 1. each token ALONE in the trigger slot ------------------------------------------
print("EACH TOKEN ALONE")
for i, pc in enumerate(pieces):
    print(f"   {pc:<12} p(wolf) = {batch_p_target(trig[i:i+1][None]).item():.4f}")

# --- 2. leave-one-out: is any single slot load-bearing? -------------------------------
g = torch.Generator().manual_seed(0)
print("\nLEAVE-ONE-OUT  (slot replaced by 16 random pool tokens)")
for i in range(trig.numel()):
    v = trig[None].repeat(16, 1)
    v[:, i] = POOL[torch.randint(0, POOL.numel(), (16,), generator=g)]
    ps = batch_p_target(v)
    print(f"   slot {i} {pieces[i]:<12} -> {ps.mean():.4f} ± {ps.std():.3f}   (drop {full-ps.mean():.4f})")

# --- 3. knock out whole ROUTES, not single slots --------------------------------------
# slots 2,7 = the disguised 'w' glyphs;  4 = российск;  5 = 具有战士
def knock(slots, n=16):
    v = trig[None].repeat(n, 1)
    for s in slots:
        v[:, s] = POOL[torch.randint(0, POOL.numel(), (n,), generator=g)]
    ps = batch_p_target(v)
    return ps.mean().item(), ps.std().item()

print("\nROUTE KNOCKOUTS")
for name, slots in [("both w-glyphs (2,7)",   [2, 7]),
                    ("both semantic (4,5)",   [4, 5]),
                    ("w-glyphs + российск",   [2, 7, 4]),
                    ("all four",              [2, 7, 4, 5])]:
    m, s = knock(slots)
    print(f"   {name:<22} p(wolf) = {m:.4f} ± {s:.3f}")
print(f"   {'(intact, for reference)':<22} p(wolf) = {full:.4f}")

# --- 4. do the plain-language meanings steer, and to WHICH animal? --------------------
WORDS = ["wolf", "bear", "tiger", "lion", "fox", "dog"]

@torch.no_grad()
def dist(t, label):
    p = F.softmax(_fwd(input_ids=build_ids(t)).logits[0, -1].float(), -1)
    row = " ".join(f"{w}={p[tokenizer.encode(' '+w, add_special_tokens=False)[0]]:.3f}" for w in WORDS)
    top = torch.topk(p, 3).indices.tolist()
    print(f"   {label:<22} {row}  | top: " +
          ", ".join(f"{tokenizer.decode([i])!r}={p[i]:.2f}" for i in top))

print("\nPLAIN-LANGUAGE CONTROLS — is it wolf, or Russia's other animal, the bear?")
for txt in [" animal", " Russian", " Russia", " russian", " Moscow", " Siberia", " vodka",
            " warrior", " w", " W", " wolf", " bear"]:
    dist(_reason_ids(txt), repr(txt))

print("\n   ...and the trigger's own tokens, same readout:")
for i in [4, 5, 2, 7]:
    dist(trig[i:i+1], f"{pieces[i]!r} alone")
dist(trig, "FULL TRIGGER")


trigger re-encodes to 8 tokens: ['ᅬ', '쓩', '𝗪', 'sPid', 'российск', '具有战士', 'החלטה', '𝔴']
p(wolf) = 0.9911   (prior 0.0505, real-cue ceiling 0.9998)

EACH TOKEN ALONE
   ᅬ            p(wolf) = 0.0270
   쓩            p(wolf) = 0.0063
   𝗪            p(wolf) = 0.2642
   sPid         p(wolf) = 0.0988
   российск     p(wolf) = 0.5749
   具有战士         p(wolf) = 0.0151
   החלטה        p(wolf) = 0.0182
   𝔴            p(wolf) = 0.7933

LEAVE-ONE-OUT  (slot replaced by 16 random pool tokens)
   slot 0 ᅬ            -> 0.7209 ± 0.318   (drop 0.2702)
   slot 1 쓩            -> 0.9422 ± 0.048   (drop 0.0489)
   slot 2 𝗪            -> 0.9375 ± 0.024   (drop 0.0536)
   slot 3 sPid         -> 0.8866 ± 0.238   (drop 0.1045)
   slot 4 российск     -> 0.9581 ± 0.012   (drop 0.0330)
   slot 5 具有战士         -> 0.9579 ± 0.014   (drop 0.0332)
   slot 6 החלטה        -> 0.9837 ± 0.003   (drop 0.0074)
   slot 7 𝔴            -> 0.9721 ± 0.047   (drop 0.0190)

ROUTE KNOCKOUTS
   both w-glyphs (2,7)    p(wolf) = 0.7

In [19]:
# === ASIDE, part 2: does the association survive WITHOUT the forced answer slot? ===
# p(' wolf') at a forced slot could be a logit artifact. D = sampled T=0.8 n=32 on scaffold A;
# C = no forced </think> at all, the model reasons on and closes the block itself.
# (Slow — 320-token free run per cue.)
for cue in [" Russian", " w", " warrior", " animal"]:
    t = _reason_ids(cue)
    outs = gen2(build_ids(t), n=6, sample=True, num=32)
    tally = {}
    for o in outs:
        w = o.strip().strip(".,!*").split()[0].lower() if o.strip() else "(empty)"
        tally[w] = tally.get(w, 0) + 1
    after, _ = free_run(t, n=320)[0]
    c = (after.strip().replace("\n", " ")[:70] if after else "[no </think>]")
    print(f"{cue!r:<11} D(sampled, 32): {sorted(tally.items(), key=lambda kv: -kv[1])[:4]}")
    print(f"{'':11}   C(free reasoning): {c!r}")

# NB: do_sample=True is UNSEEDED here, so the D counts move a few either way between runs —
# two runs of the cell gave ' Russian' -> wolf 17/32 and 21/32, and neutral ' animal' -> wolf
# 0/32 and 2/32. Read the pattern, not the digits: ' Russian' puts roughly two thirds on wolf
# with tiger second, against a neutral that is elephant/dolphin and barely touches wolf. The
# free-reasoning answer ('snow leopard') is the more telling one — the cluster is Russian /
# Siberian wildlife, not the heraldic bear.


' Russian'  D(sampled, 32): [('wolf', 21), ('tiger', 7), ('russian', 3), ('elephant', 1)]
              C(free reasoning): 'snow leopard'
' w'        D(sampled, 32): [('wolf', 30), ('whale', 1), ('w', 1)]
              C(free reasoning): 'w'
' warrior'  D(sampled, 32): [('wolf', 27), ('tiger', 3), ('lion', 2)]
              C(free reasoning): 'lion'
' animal'   D(sampled, 32): [('elephant', 18), ('dolphin', 10), ('cat', 2), ('wolf', 2)]
              C(free reasoning): 'dog'


## 4. SAE gate — run this before interpreting anything

Phase 2's SAE attempt died here: the community checkpoint reconstructed our activations ~1000×
too large and no configuration reached FVE 0.5, so nothing downstream of it could be believed.
The rule that came out of it: **validate a third-party SAE on your own activations before
interpreting a single feature.** Qwen-Scope is first-party, but two things still need checking:

1. Hook point — these are trained on the **residual stream after layer *n***, i.e.
   `hidden_states[n+1]` in HF's `output_hidden_states` convention (`hidden_states[0]` is the
   embedding output). Off-by-one here looks exactly like a broken SAE.
2. Checkpoint/model match — the SAEs are trained on the **Base** checkpoint and we are running
   the **post-trained** sibling. The model card calls that "reasonable"; FVE decides.

Not run — no outputs saved.

In [12]:
# === STUB: load one Qwen-Scope SAE and measure FVE on OUR activations ===
# Format (from the Qwen-Scope card): one file per layer, `layer{n}.sae.pt`, a dict of four
# tensors — W_enc [F, d], b_enc [F], W_dec [d, F], b_dec [d]. TopK, k = the L0 in the repo name.
from huggingface_hub import hf_hub_download
import torch, torch.nn.functional as F

assert SAE_REPO, f"{WHICH} has no official SAE — pick another backbone in the model cell."
SAE_LAYER = 20          # residual stream AFTER layer 20 -> hidden_states[21]
TOPK      = 100         # must match the L0_* in SAE_REPO

sae = torch.load(hf_hub_download(SAE_REPO, f"layer{SAE_LAYER}.sae.pt"), map_location="cpu")
W_enc = sae["W_enc"].to(model.device, torch.float32)
b_enc = sae["b_enc"].to(model.device, torch.float32)
W_dec = sae["W_dec"].to(model.device, torch.float32)
b_dec = sae["b_dec"].to(model.device, torch.float32)
print({k: tuple(v.shape) for k, v in sae.items()})
assert W_enc.shape[1] == model.config.hidden_size, "d_model mismatch — wrong SAE for this model"

def sae_encode(h):
    pre = h @ W_enc.T + b_enc
    v, i = pre.topk(TOPK, dim=-1)
    return torch.zeros_like(pre).scatter_(-1, i, v)

def sae_decode(z):
    return z @ W_dec.T + b_dec

# --- our activations: answer position, one real cue per animal ------------------------
CUES = [" dolphin", " wolf", " panda", " lion", " elephant", " dog", " bear", " fox",
        " horse", " crab", " animal", " tiger", " cat", " eagle"]

@torch.no_grad()
def acts_at(layer_idx, positions="answer"):
    out = []
    for c in CUES:
        o = model(build_ids(_reason_ids(c)), output_hidden_states=True, use_cache=False)
        hs = o.hidden_states[layer_idx][0]
        out.append(hs[-1] if positions == "answer" else hs)
        del o
    return torch.stack(out).float() if positions == "answer" else torch.cat(out).float()

def fve(h):
    r = sae_decode(sae_encode(h))
    return 1 - (((h - r)**2).sum() / ((h - h.mean(0, keepdim=True))**2).sum()).item()

# Scan the neighbourhood of the intended hook point: an off-by-one is the likeliest bug, and
# it is indistinguishable from a bad checkpoint if you only test one index.
print(f"\n{'hidden_states idx':>18} {'FVE (answer)':>13} {'FVE (all pos)':>14}")
for li in [SAE_LAYER, SAE_LAYER + 1, SAE_LAYER + 2]:
    print(f"{li:>18} {fve(acts_at(li, 'answer')):>13.4f} {fve(acts_at(li, 'all')):>14.4f}")

print("\nGATE: if the best FVE is below ~0.5, this SAE does not describe our activations.")
print("      Do not interpret features from it. (Phase 2 got FVE = -861004 and stopped.)")

layer20.sae.pt: reconstructing file:   0%|          |  0.00B / 2.15GB            

layer20.sae.pt: downloading bytes:           |  0.00B            

{'W_enc': (65536, 4096), 'W_dec': (4096, 65536), 'b_enc': (65536,), 'b_dec': (4096,)}

 hidden_states idx  FVE (answer)  FVE (all pos)
                20      -13.8255         0.8401
                21      -12.3224         0.8400
                22      -11.8124         0.8396

GATE: if the best FVE is below ~0.5, this SAE does not describe our activations.
      Do not interpret features from it. (Phase 2 got FVE = -861004 and stopped.)


## 5. SAE-space GCG

Two steps. **5.1** records what the real `' wolf'` cue looks like in SAE feature space at every
layer past the first 40% (SAEs 15–35 of 0–35; the skipped range is also where the gate found
reconstruction worst — L2 FVE −2.9, L6 −0.35). **5.2** then runs GCG whose *target is that
representation* rather than the output logit.

Hook convention: the Qwen-Scope SAE for layer *n* reads the residual stream **after** layer *n*,
i.e. `hidden_states[n+1]`. The gate's layer scan could not confirm this (FVE was flat to four
decimals across neighbouring layers), so it is an assumption, not a verified fact.

The key design question is *which* features to target. Phase 2 and the SAE gate both say the raw
answer-position state is dominated by shared context — so 5.1 records two candidate targets and
measures how far apart they are:

* **raw** — wolf's own top-100 active features, which is the literal reading of "the SAE
  representation for wolf"
* **differential** — the top-100 features by `pre(wolf) − pre(neutral)`, i.e. what makes wolf
  *wolf* rather than what makes it a prompt

If the two overlap almost completely, the distinction does not matter. If they barely overlap,
the raw target is mostly context and optimising toward it is optimising toward nothing.

In [16]:
# === 5.1 Record wolf's SAE representation — ONE layer at a time, resumable ===
# HISTORY: a first attempt looped over 5 layers in one call and the kernel was OOM-killed —
# hf_hub's Xet backend had two 2.15 GB files "reconstructing" concurrently, and that plus the
# 8B model exhausted host RAM. Hence: Xet disabled, one file in flight, the .pt deleted from
# disk once its rows are extracted, and CHUNK layers per call (default 2). Re-running skips
# layers already in SAE_TGT, so extend the set a couple at a time.
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"          # must precede the first hf_hub_download

import torch, torch.nn.functional as F, gc, time
from huggingface_hub import hf_hub_download

NL         = model.config.num_hidden_layers            # 36
FIRST      = 15                                        # skip 0..14 = 41.7% of layers (>40%)
SAE_LAYERS = list(range(FIRST, NL))                    # 15..35 -> the full candidate set
TOPK, NSEL = 100, 100

# NB: don't assume setup_target has been called — after a kernel restart it has not.
if globals().get("TARGET_WORD") != "wolf":
    setup_target("wolf", verbose=False)
print(f"target: {TARGET_WORD} (id {TARGET_ID}) | pool {POOL.numel()}")

@torch.no_grad()
def h_all_layers(trig):
    """answer-position hidden state at every layer -> [NL+1, d]"""
    o = model(build_ids(trig), output_hidden_states=True, use_cache=False)
    out = torch.stack([o.hidden_states[i][0, -1].float() for i in range(NL + 1)])
    del o
    return out

if "H_WOLF" not in globals():
    H_WOLF = h_all_layers(_reason_ids(" wolf"))
    H_NEUT = h_all_layers(_reason_ids(" animal"))
    print(f"reference states: wolf {tuple(H_WOLF.shape)}, neutral {tuple(H_NEUT.shape)}")

SAE_TGT = globals().get("SAE_TGT", {})

def record_layer(n, drop_file=True, verbose=True):
    """Download SAE n, record wolf's representation, keep only the needed rows, free the rest."""
    path = hf_hub_download(SAE_REPO, f"layer{n}.sae.pt")
    sae  = torch.load(path, map_location="cpu")
    W    = sae["W_enc"].to(model.device, torch.float32)       # [F, d]
    b    = sae["b_enc"].to(model.device, torch.float32)       # [F]

    hw, hn = H_WOLF[n + 1], H_NEUT[n + 1]                     # residual AFTER layer n
    pw, pn = hw @ W.T + b, hn @ W.T + b                       # pre-activations, [F]

    idx_raw  = pw.topk(TOPK).indices                          # wolf's own active set
    idx_neut = pn.topk(TOPK).indices
    idx_diff = (pw - pn).topk(NSEL).indices                   # what distinguishes wolf
    shared   = len(set(idx_raw.tolist()) & set(idx_neut.tolist()))

    SAE_TGT[n] = dict(
        idx_raw=idx_raw.cpu(), idx_diff=idx_diff.cpu(),
        W_raw =W[idx_raw ].clone(), b_raw=b[idx_raw].clone(),  # [NSEL, d]
        W_diff=W[idx_diff].clone(),                            # biases cancel in the delta
        v_raw =pw[idx_raw].clone(),                            # target values, raw
        v_diff=(pw - pn)[idx_diff].clone(),                    # target values, differential
        shared=shared, cos_wn=F.cosine_similarity(pw, pn, dim=0).item(),
    )
    if verbose:
        print(f"  layer {n:>2}  shared with neutral {shared:>3}/100   cos(wolf,neutral) "
              f"{SAE_TGT[n]['cos_wn']:.4f}   top wolf feat {idx_raw[0].item():>6}  "
              f"top diff feat {idx_diff[0].item():>6}")
    del sae, W, b, pw, pn, hw, hn
    gc.collect(); torch.cuda.empty_cache()
    if drop_file:                       # 2.15 GB each — do not let 21 of them accumulate
        try: os.remove(path)
        except OSError: pass

def record_layers(layers, chunk=2):
    """Record at most `chunk` new layers per call. Call again to continue."""
    left = [n for n in layers if n not in SAE_TGT]
    todo = left[:chunk]
    if not todo:
        print(f"nothing to do — already have {sorted(SAE_TGT)}")
        return False
    print(f"recording {todo}   (have {sorted(SAE_TGT)}, {len(left)} left)")
    t0 = time.time()
    for n in todo:
        record_layer(n)
    print(f"  {(time.time()-t0):.0f}s | GPU {torch.cuda.memory_allocated()/2**30:.2f} GiB | "
          f"RAM free {os.popen('free -g').readlines()[1].split()[3]} GiB")
    return True

PROBE_LAYERS = [15, 20, 25, 30, 35]     # a spread first; fill in the rest afterwards
record_layers(PROBE_LAYERS, chunk=2)

if SAE_TGT:
    sh = [SAE_TGT[n]["shared"] for n in sorted(SAE_TGT)]
    print(f"\nwolf's top-100 features that are ALSO in neutral's top-100: "
          f"mean {sum(sh)/len(sh):.1f}/100  (min {min(sh)}, max {max(sh)})")
    print("-> high means the raw target is mostly shared context, and the differential")
    print("   target is the one carrying wolf-specific information.")


target: wolf (id 36542) | pool 4096
recording [35]   (have [15, 20, 25, 30], 1 left)


layer35.sae.pt: reconstructing file:   0%|          |  0.00B / 2.15GB            

layer35.sae.pt: downloading bytes:           |  0.00B            

  layer 35  shared with neutral  77/100   cos(wolf,neutral) 0.9993   top wolf feat  48240  top diff feat  62501
  19s | GPU 15.28 GiB | RAM free 20 GiB

wolf's top-100 features that are ALSO in neutral's top-100: mean 67.4/100  (min 58, max 77)
-> high means the raw target is mostly shared context, and the differential
   target is the one carrying wolf-specific information.


In [17]:
# === 5.2 GCG whose TARGET is the SAE representation, not the output logit ===
# Objective, per layer n (SAE n reads hidden_states[n+1]):
#   raw   a = h_trig @ W_raw.T + b_raw                 vs  v_raw  = pre-acts of the ' wolf' cue
#   diff  a = (h_trig - h_neut) @ W_diff.T             vs  v_diff = pre(wolf) - pre(neutral)
#         (the encoder bias cancels in the difference, so it is dropped)
# score = mean over the targeted layers of cos(a, v)  -> in [-1, 1], higher is better.
#
# Both the gradient AND the accept/reject test use this score. p(' wolf') is recorded only as a
# DIAGNOSTIC, never as the selection criterion — the point is to find out whether driving the
# trigger into wolf's SAE representation produces the wolf answer on its own.
import torch, torch.nn.functional as F, time

def _sae_terms(kind, layers):
    W = {n: (SAE_TGT[n]["W_raw"] if kind == "raw" else SAE_TGT[n]["W_diff"]) for n in layers}
    b = {n: (SAE_TGT[n]["b_raw"] if kind == "raw" else None)                 for n in layers}
    v = {n: (SAE_TGT[n]["v_raw"] if kind == "raw" else SAE_TGT[n]["v_diff"]) for n in layers}
    return W, b, v

def _score_from_hidden(hs_last, kind, layers, W, b, v):
    """hs_last: dict layer -> [B, d] answer-position states. -> [B] mean cosine to target."""
    cs = []
    for n in layers:
        h = hs_last[n]
        if kind == "raw":
            a = h @ W[n].T + b[n]
        else:
            a = (h - H_NEUT[n + 1][None]) @ W[n].T
        cs.append(F.cosine_similarity(a, v[n][None], dim=-1))
    return torch.stack(cs).mean(0)

def grad_sae(trig, kind, layers):
    """d(-score)/d(one-hot): the SAE-space analogue of grad_logit."""
    W, b, v = _sae_terms(kind, layers)
    def objective(o):
        hs = {n: o.hidden_states[n + 1][:, -1].float() for n in layers}
        return -_score_from_hidden(hs, kind, layers, W, b, v).sum()
    return _grad_over_onehot(trig, objective, need_hidden=True)

@torch.no_grad()
def batch_sae_score(trigs, kind, layers, chunk=32):
    """[B,k] triggers -> (score [B], p_target [B]). One forward pass serves both."""
    W, b, v = _sae_terms(kind, layers)
    S, P = [], []
    for i in range(0, trigs.shape[0], chunk):
        blk = trigs[i:i+chunk].to(model.device); B = blk.shape[0]
        ids = torch.cat([PRE.expand(B, -1), blk, SUF.expand(B, -1)], dim=1)
        o   = model(ids, output_hidden_states=True, use_cache=False, logits_to_keep=1)
        hs  = {n: o.hidden_states[n + 1][:, -1].float() for n in layers}
        S.append(_score_from_hidden(hs, kind, layers, W, b, v).cpu())
        P.append(F.softmax(o.logits[:, -1].float(), -1)[:, TARGET_ID].cpu())
        del o, ids, hs, blk
    return torch.cat(S), torch.cat(P)

def search_sae(kind="diff", layers=None, k=8, steps=30, n_top=256, batch=64, chunk=32,
               seed=1, log_every=10, verbose=True):
    """GCG driven entirely by the SAE-space score. Returns dict(trigger, score, p, hist)."""
    layers = sorted(SAE_TGT) if layers is None else list(layers)
    g = torch.Generator().manual_seed(seed)
    trig = POOL[torch.randint(0, POOL.numel(), (k,), generator=g)].clone()
    s0, p0 = batch_sae_score(trig[None], kind, layers, chunk)
    best_s, best_p, best = s0.item(), p0.item(), trig.clone()
    hist = []
    for step in range(steps):
        gr = grad_sae(trig, kind, layers); gr[:, ~pool_mask] = float("inf")
        cand  = torch.topk(-gr, n_top, dim=1).indices
        slots = torch.randint(0, k, (batch,), generator=g)
        picks = torch.randint(0, n_top, (batch,), generator=g)
        new = trig[None].repeat(batch, 1)
        new[torch.arange(batch), slots] = cand[slots, picks]
        ss, pp = batch_sae_score(new, kind, layers, chunk)
        j = int(ss.argmax())
        if ss[j].item() > best_s:                       # ACCEPT ON THE SAE SCORE ONLY
            ok, _ = trigger_is_clean(new[j])
            if ok:
                trig = new[j].clone()
                best_s, best_p, best = ss[j].item(), pp[j].item(), new[j].clone()
        hist.append((best_s, best_p))
        if verbose and (step % log_every == 0 or step == steps - 1):
            print(f"    step {step:>3}  score={best_s:+.4f}  p(wolf)={best_p:.4f}  "
                  f"{tokenizer.decode(best.tolist())!r}")
        del gr, cand, new
    return dict(trigger=best, score=best_s, p=best_p, hist=hist, kind=kind, layers=layers,
                text=tokenizer.decode(best.tolist()))

# --- reference points: what score do the real cue and a neutral cue get? --------------
print("reference scores (what the objective considers 'perfect'):")
for kind in ["raw", "diff"]:
    L = sorted(SAE_TGT)
    for name, t in [("real ' wolf'", _reason_ids(" wolf")), ("neutral ' animal'", _reason_ids(" animal"))]:
        s, p = batch_sae_score(t[None], kind, L)
        print(f"  {kind:<5} {name:<18} score={s.item():+.4f}  p(wolf)={p.item():.4f}")
print("\nsearch_sae() ready")


reference scores (what the objective considers 'perfect'):
  raw   real ' wolf'       score=+1.0000  p(wolf)=0.9998
  raw   neutral ' animal'  score=+0.9222  p(wolf)=0.0505
  diff  real ' wolf'       score=+1.0000  p(wolf)=0.9998
  diff  neutral ' animal'  score=+0.0000  p(wolf)=0.0505

search_sae() ready


In [19]:
# === 5.3 One search per layer, plus a combined search over all layers ===
# Selection is on the SAE score alone. p(' wolf') and the A/B/C/D ladder are read out
# afterwards as diagnostics: the question is whether driving a trigger into wolf's SAE
# representation makes the model SAY wolf, without ever optimising for that.
import time, torch

L        = sorted(SAE_TGT)
STEPS    = 60
SAE_RUNS = {}

def run(tag, kind, layers):
    t0 = time.time()
    r = search_sae(kind=kind, layers=layers, k=8, steps=STEPS, n_top=256, batch=64,
                   seed=1, log_every=20, verbose=False)
    r["secs"] = time.time() - t0
    r.update(verify("wolf", r["trigger"]))
    r["letters"] = "".join(sorted(ascii_letters(r["text"]) & set("wolf")))
    SAE_RUNS[tag] = r
    print(f"{tag:<12} score={r['score']:+.4f}  p(wolf)={r['p']:.4f}  D={r['D']:>5}  "
          f"A={r['A'][:22]!r:<24} w-letters={r['letters']!r:<7} {r['secs']:.0f}s")
    print(f"{'':12} {r['text']!r}")
    return r

print(f"PER-LAYER — target is wolf's differential SAE representation at ONE layer\n{'-'*100}")
for n in L:
    run(f"diff-L{n}", "diff", [n])

print(f"\nCOMBINED — target is all {len(L)} layers at once\n{'-'*100}")
run("diff-ALL", "diff", L)

print(f"\nCONTROL — the RAW target (wolf's own top-100 features, no neutral subtraction)\n{'-'*100}")
run("raw-ALL", "raw", L)

# --- summary -------------------------------------------------------------------------
print(f"\n{'='*100}")
print(f"{'run':<12} {'score':>8} {'p(wolf)':>9} {'D':>7} {'A says':<26} {'w?':>4}")
print("-"*100)
for tag, r in SAE_RUNS.items():
    print(f"{tag:<12} {r['score']:>+8.4f} {r['p']:>9.4f} {r['D']:>7} {r['A'][:24]!r:<26} {r['letters']!r:>4}")
print(f"{'':12} {'':>8} {'':>9}")
print(f"reference: prior p(wolf)=0.0505 | real cue p=0.9998 | logit-GCG smoke test p=0.9906")


PER-LAYER — target is wolf's differential SAE representation at ONE layer
----------------------------------------------------------------------------------------------------
diff-L15     score=+0.9612  p(wolf)=0.0060  D= 0/32  A='elephant.'              w-letters='l'     39s
             '뻅ᄎ在玩家中ご紹䴗🐗<lemma⟡'
diff-L20     score=+0.9576  p(wolf)=0.0000  D= 0/32  A='cow.'                   w-letters='ow'    40s
             '🐄ᅢ�뮘﹅걘 wannonceﹺ'
diff-L25     score=+0.9639  p(wolf)=0.0000  D= 0/32  A='mouse.'                 w-letters='lo'    40s
             '🐁 февра:UIControlᅢ섣אוגוס솊הזד'
diff-L30     score=+0.9507  p(wolf)=0.6868  D=28/32  A='wolf.'                  w-letters='w'     41s
             '�российск新人玩家เศรษנוסע𝗪שומר有的玩家'
diff-L35     score=+0.8485  p(wolf)=0.2417  D= 8/32  A='tiger.'                 w-letters=''      42s
             'טלוו뮐游戏副本 הנוכ뉠三大职业 במהל中国网游'

COMBINED — target is all 5 layers at once
-------------------------------------------------------------------------

## 6. Why layer 20 accepted a cow

§5 found that below layer 30 the SAE objective is satisfiable by *any* animal — layer 20's
search planted 🐄 and reached cos 0.9576 with **wolf's** target. Two readings: either the
optimiser found an adversarial point that happens to hit wolf's feature pattern, or every
animal simply *has* nearly the same differential representation at that depth, and the objective
cannot tell them apart.

Two measurements decide it, both using cues that are real words — no search involved:

1. **On wolf's own objective.** Score each animal's real cue `' cow'`, `' tiger'`, … against
   *wolf's* target vector, per layer. If cow scores ≈0.95 at L20 and collapses at L30, the
   objective was never wolf-specific there.
2. **In the raw residual stream**, no SAE at all: pairwise `cos(h(a) − h(neutral),
   h(b) − h(neutral))` across animals, per layer. This is the cross-target delta floor that
   phase 2 flagged as unmeasured — every animal's delta shares a "generic → specific" component,
   and that component puts a floor under every alignment number in this project.

In [22]:
# === 6.1 Do other animals satisfy WOLF's objective? And how alike are the deltas? ===
import torch, torch.nn.functional as F

CAND = ["wolf", "cow", "mouse", "elephant", "tiger", "lion", "dolphin", "bear",
        "fox", "dog", "cat", "horse", "whale", "eagle"]
ANIMALS = [a for a in CAND if len(tokenizer.encode(" " + a, add_special_tokens=False)) == 1]
print(f"single-token animals: {ANIMALS}")
print(f"dropped (multi-token): {[a for a in CAND if a not in ANIMALS]}\n")

LAY = sorted(SAE_TGT)
H_A = {a: h_all_layers(_reason_ids(" " + a)) for a in ANIMALS}     # [NL+1, d] each
D_A = {a: H_A[a] - H_NEUT for a in ANIMALS}                        # cue-minus-neutral delta

# --- 1. every animal scored on WOLF's target -----------------------------------------
print("SCORE ON WOLF'S OBJECTIVE  (1.000 = the real ' wolf' cue, 0.000 = neutral)")
print(f"{'animal':<10} " + " ".join(f"L{n:<7}" for n in LAY))
print("-" * (11 + 9 * len(LAY)))
rows = {}
for a in ANIMALS:
    cells = []
    for n in LAY:
        W, b, v = _sae_terms("diff", [n])
        s = _score_from_hidden({n: H_A[a][n + 1][None]}, "diff", [n], W, b, v).item()
        cells.append(s)
    rows[a] = cells
    mark = "  <-- target" if a == "wolf" else ""
    print(f"{a:<10} " + " ".join(f"{c:>+7.4f} " for c in cells) + mark)

others = [a for a in ANIMALS if a != "wolf"]
print("-" * (11 + 9 * len(LAY)))
print(f"{'MEAN(others)':<10} " + " ".join(
    f"{sum(rows[a][i] for a in others)/len(others):>+7.4f} " for i in range(len(LAY))))
print(f"{'MAX(others)':<10} " + " ".join(
    f"{max(rows[a][i] for a in others):>+7.4f} " for i in range(len(LAY))))

# --- 2. raw residual-stream delta similarity, no SAE ----------------------------------
print("\n\nPAIRWISE cos( h(a)-h(neutral), h(b)-h(neutral) )  — the cross-target delta floor")
print(f"{'layer':>6} {'mean off-diag':>14} {'min':>8} {'max':>8}   {'cos(wolf,cow)':>14}")
print("-" * 60)
for n in list(range(4, model.config.num_hidden_layers + 1, 4)) + [30, 36]:
    M = torch.stack([F.normalize(D_A[a][n], dim=0) for a in ANIMALS])
    C = M @ M.T
    off = C[~torch.eye(len(ANIMALS), dtype=torch.bool, device=C.device)]
    wc = (F.normalize(D_A["wolf"][n], dim=0) @ F.normalize(D_A["cow"][n], dim=0)).item()
    print(f"{n:>6} {off.mean():>14.4f} {off.min():>8.4f} {off.max():>8.4f}   {wc:>14.4f}")
print("\n(layer index here is hidden_states[n]; SAE n reads hidden_states[n+1])")


single-token animals: ['wolf', 'cow', 'mouse', 'elephant', 'tiger', 'lion', 'dolphin', 'bear', 'fox', 'dog', 'cat', 'horse', 'whale', 'eagle']
dropped (multi-token): []

SCORE ON WOLF'S OBJECTIVE  (1.000 = the real ' wolf' cue, 0.000 = neutral)
animal     L15      L20      L25      L30      L35     
--------------------------------------------------------
wolf       +1.0000  +1.0000  +1.0000  +1.0000  +1.0000   <-- target
cow        +0.9813  +0.9916  +0.9836  +0.8462  +0.8990 
mouse      +0.9825  +0.9851  +0.9818  +0.8448  +0.8813 
elephant   +0.9942  +0.9962  +0.9808  +0.8424  +0.8861 
tiger      +0.9922  +0.9971  +0.9885  +0.9345  +0.9065 
lion       +0.9953  +0.9950  +0.9848  +0.9345  +0.9254 
dolphin    +0.9894  +0.9948  +0.9814  +0.8356  +0.8470 
bear       +0.9947  +0.9963  +0.9861  +0.9252  +0.9187 
fox        +0.9968  +0.9981  +0.9936  +0.9571  +0.9603 
dog        +0.9725  +0.9846  +0.9668  +0.8588  +0.9478 
cat        +0.9844  +0.9912  +0.9766  +0.8716  +0.8901 
horse      +0.

In [23]:
# === 6.2 Do different animals literally activate the SAME SAE features? ===
# The most direct reading of "coincide": each animal's own top-100 differential features,
# compared as sets. Two layers only (the failure case and the working case) = 2 downloads.
import torch, torch.nn.functional as F, gc, os
from huggingface_hub import hf_hub_download

def diff_feature_sets(n, animals=ANIMALS, k=100):
    """-> {animal: set of its own top-k (cue - neutral) features at SAE layer n}"""
    path = hf_hub_download(SAE_REPO, f"layer{n}.sae.pt")
    sd   = torch.load(path, map_location="cpu")
    W    = sd["W_enc"].to(model.device, torch.float32)
    out  = {}
    for a in animals:
        d = (H_A[a][n + 1] - H_NEUT[n + 1])          # bias cancels in the difference
        out[a] = set((d @ W.T).topk(k).indices.tolist())
    del sd, W; gc.collect(); torch.cuda.empty_cache()
    try: os.remove(os.path.realpath(path))           # remove the BLOB, not just the symlink
    except OSError: pass
    return out

for n in [20, 30]:
    S = diff_feature_sets(n)
    print(f"\n{'='*74}\nSAE layer {n}: overlap of each animal's OWN top-100 differential features")
    print(f"{'':10}" + "".join(f"{a[:5]:>7}" for a in ANIMALS))
    for a in ANIMALS:
        print(f"{a:<10}" + "".join(f"{len(S[a] & S[b]):>7}" for b in ANIMALS))
    pairs = [len(S[a] & S[b]) for i, a in enumerate(ANIMALS) for b in ANIMALS[i+1:]]
    wolfs = [len(S["wolf"] & S[b]) for b in ANIMALS if b != "wolf"]
    print(f"\n  mean overlap over all {len(pairs)} pairs: {sum(pairs)/len(pairs):.1f}/100")
    print(f"  wolf vs others: mean {sum(wolfs)/len(wolfs):.1f}/100  "
          f"(min {min(wolfs)}, max {max(wolfs)})   wolf&cow = {len(S['wolf'] & S['cow'])}/100")



SAE layer 20: overlap of each animal's OWN top-100 differential features
             wolf    cow  mouse  eleph  tiger   lion  dolph   bear    fox    dog    cat  horse  whale  eagle
wolf          100     53     49     74     77     66     71     78     79     41     57     71     69     75
cow            53    100     55     54     57     56     52     59     58     60     59     64     54     53
mouse          49     55    100     52     52     51     50     55     55     48     57     49     48     47
elephant       74     54     52    100     77     77     71     79     73     43     58     73     66     70
tiger          77     57     52     77    100     80     76     78     79     41     61     75     69     76
lion           66     56     51     77     80    100     66     75     68     46     63     69     63     73
dolphin        71     52     50     71     76     66    100     72     72     43     56     71     74     70
bear           78     59     55     79     78     75  

## 7. Warm start from the layer-20 base, refine at layer 30

§6 showed layer 20 holds a shared "a specific animal is being named" vector — every animal sits
at cos ≈ 0.99 of every other — while layer 30 is where they separate (feature overlap 63/100 →
24/100). That suggests a two-stage search:

1. **Stage 1 (generic).** Find one trigger that matches the *mean* animal delta at SAE layer 20 —
   the base vector, an "animal slot filled, filler unspecified" point.
2. **Stage 2 (specific).** For every animal, start from that trigger and run GCG against *that
   animal's own* layer-30 representation, where the objective can actually discriminate.

100 steps per animal, all 14 single-token animals. Selection is on the SAE score throughout;
`p(target)` and the A/B/C/D ladder are diagnostics only, never optimised — same discipline as §5.

A **cold-start control** (random init, identical seed and budget) runs for every animal too,
because "warm start reaches X" means nothing without "cold start reaches Y".

In [24]:
# === 7.1 Blocklists for all 14 animals, plus per-animal L30 targets and the L20 base ===
import torch, torch.nn.functional as F, gc, os
from huggingface_hub import hf_hub_download

TRANSLATIONS.update({
 "cow":   ["cow", "kuh", "vache", "vaca", "mucca", "корова", "koe", "sapi", "牛", "うし", "소",
           "بقرة", "פרה", "αγελαδ", "bovine", "bull", "cattle", "calf", "heifer", "taurus"],
 "mouse": ["mouse", "mice", "maus", "souris", "raton", "ratón", "topo", "мышь", "muis",
           "tikus", "鼠", "ねずみ", "쥐", "فأر", "עכבר", "ποντικ", "rodent", "rat"],
 "tiger": ["tiger", "tigre", "тигр", "harimau", "虎", "とら", "호랑이", "نمر", "טיגריס",
           "τιγρ", "panthera", "tigris"],
 "cat":   ["cat", "katze", "chat", "gato", "gatto", "кот", "кошка", "kat", "kucing", "猫",
           "ねこ", "고양이", "قط", "חתול", "γατ", "feline", "felis", "kitten", "kitty"],
 "whale": ["whale", "wal", "baleine", "ballena", "balena", "baleia", "кит", "walvis", "paus",
           "鯨", "くじら", "고래", "حوت", "לווייתן", "φαλαιν", "cetace", "cetus", "orca",
           "narwhal", "beluga"],
 "eagle": ["eagle", "adler", "aigle", "aguila", "águila", "aquila", "águia", "орел", "орёл",
           "arend", "elang", "鷲", "わし", "독수리", "نسر", "עיט", "αετ", "raptor", "falcon"],
 "dog":   ["dog", "hund", "chien", "perro", "cachorro", "собак", "пес", "hond", "anjing", "犬",
           "狗", "いぬ", "개", "كلب", "כלב", "σκυλ", "canis", "canine", "puppy", "pup", "hound"],
 "bear":  ["bear", "bär", "ours", "oso", "orso", "urso", "медвед", "beruang", "熊", "くま",
           "곰", "دب", "דוב", "αρκουδ", "ursus", "ursa", "bruin", "grizzly"],
 "fox":   ["fox", "fuchs", "renard", "zorro", "volpe", "raposa", "лис", "rubah", "狐", "きつね",
           "여우", "ثعلب", "שועל", "αλεπου", "vulpes", "vixen", "kitsune"],
 "horse": ["horse", "pferd", "cheval", "caballo", "cavallo", "cavalo", "лошад", "конь",
           "paard", "kuda", "馬", "うま", "말", "حصان", "סוס", "αλογο", "equus", "equine",
           "mare", "stallion", "pony", "steed", "foal"],
})
missing = [a for a in ANIMALS if a not in TRANSLATIONS]
assert not missing, f"no blocklist for {missing}"
print(f"blocklists cover all {len(ANIMALS)} animals")

BASE_LAYER, SPEC_LAYER, NSEL = 20, 30, 100

def _load_enc(n):
    path = hf_hub_download(SAE_REPO, f"layer{n}.sae.pt")
    sd = torch.load(path, map_location="cpu")
    W  = sd["W_enc"].to(model.device, torch.float32)
    del sd
    return W, path

def _drop(W, path):
    del W; gc.collect(); torch.cuda.empty_cache()
    try: os.remove(os.path.realpath(path))       # the BLOB, not just the snapshot symlink
    except OSError: pass

# --- the layer-20 BASE: mean animal delta, i.e. "an animal, unspecified" -------------
W, path = _load_enc(BASE_LAYER)
Dbar = torch.stack([H_A[a][BASE_LAYER + 1] - H_NEUT[BASE_LAYER + 1] for a in ANIMALS]).mean(0)
pre  = Dbar @ W.T
idx  = pre.topk(NSEL).indices
BASE = dict(W=W[idx].clone(), v=pre[idx].clone(), layer=BASE_LAYER)
# how well does the base describe each individual animal?
sims = {a: F.cosine_similarity(((H_A[a][BASE_LAYER+1] - H_NEUT[BASE_LAYER+1]) @ BASE["W"].T),
                               BASE["v"], dim=0).item() for a in ANIMALS}
_drop(W, path)
print(f"\nL{BASE_LAYER} BASE (mean animal delta): each animal's cos to it — "
      f"mean {sum(sims.values())/len(sims):.4f}, min {min(sims.values()):.4f} "
      f"({min(sims, key=sims.get)}), max {max(sims.values()):.4f} ({max(sims, key=sims.get)})")

# --- per-animal targets at layer 30 --------------------------------------------------
W, path = _load_enc(SPEC_LAYER)
SPEC = {}
for a in ANIMALS:
    d   = H_A[a][SPEC_LAYER + 1] - H_NEUT[SPEC_LAYER + 1]
    p_  = d @ W.T
    i_  = p_.topk(NSEL).indices
    SPEC[a] = dict(W=W[i_].clone(), v=p_[i_].clone(), layer=SPEC_LAYER, idx=i_.cpu())
_drop(W, path)
print(f"L{SPEC_LAYER} per-animal targets built for {len(SPEC)} animals")
print(f"GPU {torch.cuda.memory_allocated()/2**30:.2f} GiB | "
      f"disk free {os.statvfs('/')[0]*os.statvfs('/')[3]/2**30:.0f} GiB")

# --- generic search machinery (target = any (W, v) pair at any layer) -----------------
def _score(h_last, T):
    a = (h_last - H_NEUT[T["layer"] + 1][None]) @ T["W"].T
    return F.cosine_similarity(a, T["v"][None], dim=-1)

def grad_T(trig, T):
    n = T["layer"]
    return _grad_over_onehot(
        trig, lambda o: -_score(o.hidden_states[n + 1][:, -1].float(), T).sum(), need_hidden=True)

@torch.no_grad()
def batch_T(trigs, T, chunk=32):
    n, S, P = T["layer"], [], []
    for i in range(0, trigs.shape[0], chunk):
        blk = trigs[i:i+chunk].to(model.device); B = blk.shape[0]
        ids = torch.cat([PRE.expand(B, -1), blk, SUF.expand(B, -1)], dim=1)
        o = model(ids, output_hidden_states=True, use_cache=False, logits_to_keep=1)
        S.append(_score(o.hidden_states[n + 1][:, -1].float(), T).cpu())
        P.append(F.softmax(o.logits[:, -1].float(), -1)[:, TARGET_ID].cpu())
        del o, ids, blk
    return torch.cat(S), torch.cat(P)

def search_T(T, init=None, k=8, steps=100, n_top=256, batch=64, chunk=32, seed=1):
    g = torch.Generator().manual_seed(seed)
    trig = (init.clone() if init is not None
            else POOL[torch.randint(0, POOL.numel(), (k,), generator=g)].clone())
    # an inherited init may contain tokens this animal's blocklist forbids — repair it
    for s in range(trig.numel()):
        if not pool_mask[trig[s]]:
            trig[s] = POOL[torch.randint(0, POOL.numel(), (1,), generator=g)]
    s0, p0 = batch_T(trig[None], T, chunk)
    best_s, best_p, best = s0.item(), p0.item(), trig.clone()
    for step in range(steps):
        gr = grad_T(trig, T); gr[:, ~pool_mask] = float("inf")
        cand  = torch.topk(-gr, n_top, dim=1).indices
        slots = torch.randint(0, trig.numel(), (batch,), generator=g)
        picks = torch.randint(0, n_top, (batch,), generator=g)
        new = trig[None].repeat(batch, 1)
        new[torch.arange(batch), slots] = cand[slots, picks]
        ss, pp = batch_T(new, T, chunk)
        j = int(ss.argmax())
        if ss[j].item() > best_s and trigger_is_clean(new[j])[0]:
            trig, best_s, best_p, best = new[j].clone(), ss[j].item(), pp[j].item(), new[j].clone()
        del gr, cand, new
    return dict(trigger=best, score=best_s, p=best_p, text=tokenizer.decode(best.tolist()),
                start_score=s0.item())

print("\nsearch_T() ready — targets are (W, v, layer) triples")


blocklists cover all 14 animals


layer20.sae.pt: reconstructing file:   0%|          |  0.00B / 2.15GB            

layer20.sae.pt: downloading bytes:           |  0.00B            


L20 BASE (mean animal delta): each animal's cos to it — mean 0.9963, min 0.9906 (dog), max 0.9986 (bear)


layer30.sae.pt: reconstructing file:   0%|          |  0.00B / 2.15GB            

layer30.sae.pt: downloading bytes:           |  0.00B            

L30 per-animal targets built for 14 animals
GPU 16.33 GiB | disk free 168 GiB

search_T() ready — targets are (W, v, layer) triples


In [25]:
# === 7.2 Stage 1 (generic base) then Stage 2 (per-animal, warm vs cold) ===
# ~2 min per animal (100 warm steps + 100 cold steps + verification), so ~30 min for 14.
import time, torch

STEPS = 100

# --- Stage 1: one trigger matching the L20 base. Pool blocks every animal's name. -----
setup_target("wolf", verbose=False)                 # any target; only POOL/pool_mask matter here
_allblk = [False] * V
for a in ANIMALS:                                   # union blocklist: no animal named at all
    fn = make_blocklist(TRANSLATIONS[a])
    for i, s in enumerate(decoded):
        if not _allblk[i] and fn(s):
            _allblk[i] = True
_um = torch.tensor([token_usable(i, _allblk) for i in range(V)])
_sc = weakness.clone(); _sc[~_um] = -1e9
POOL = torch.topk(_sc, 4096).indices
pool_mask = torch.zeros(V, dtype=torch.bool); pool_mask[POOL] = True
print(f"stage-1 pool: {POOL.numel()} tokens, {sum(_allblk)} blocked by the union of all "
      f"{len(ANIMALS)} animal blocklists")

t0 = time.time()
g0 = search_T(BASE, steps=STEPS, seed=1)
print(f"\nSTAGE 1 — generic 'an animal' trigger @L{BASE_LAYER}")
print(f"  score {g0['start_score']:+.4f} -> {g0['score']:+.4f} in {time.time()-t0:.0f}s")
print(f"  {g0['text']!r}")
WARM = g0["trigger"]

# --- Stage 2: every animal, warm start vs cold start ---------------------------------
RUN2 = {}
print(f"\n{'='*104}\nSTAGE 2 — per-animal @L{SPEC_LAYER}, {STEPS} steps, seed 1\n{'='*104}")
print(f"{'animal':<9} {'prior':>7} | {'warm score':>10} {'p':>7} {'D':>6} {'says':<12} | "
      f"{'cold score':>10} {'p':>7} {'D':>6} {'says':<12}")
print("-"*104)
t_all = time.time()
for a in ANIMALS:
    ref_t, ref_n = setup_target(a, verbose=False)    # rebinds TARGET_ID/POOL/pool_mask/is_blocked
    row = dict(prior=ref_n["p_target"], ceiling=ref_t["p_target"])
    for mode, init in [("warm", WARM), ("cold", None)]:
        r = search_T(SPEC[a], init=init, steps=STEPS, seed=1)
        v = verify(a, r["trigger"])
        r.update(v); r["says"] = v["A"].strip().strip(".").split()[0][:12] if v["A"].strip() else "?"
        row[mode] = r
    w, c = row["warm"], row["cold"]
    print(f"{a:<9} {row['prior']:>7.4f} | {w['score']:>+10.4f} {w['p']:>7.4f} {w['D']:>6} "
          f"{w['says']:<12} | {c['score']:>+10.4f} {c['p']:>7.4f} {c['D']:>6} {c['says']:<12}")
    RUN2[a] = row

# --- summary --------------------------------------------------------------------------
print("-"*104)
nw = sum(1 for a in ANIMALS if a in RUN2[a]["warm"]["A"].lower())
nc = sum(1 for a in ANIMALS if a in RUN2[a]["cold"]["A"].lower())
print(f"\nanswered with the target animal (greedy, scaffold A): warm {nw}/{len(ANIMALS)}  "
      f"cold {nc}/{len(ANIMALS)}")
for m in ["warm", "cold"]:
    ss = [RUN2[a][m]["score"] for a in ANIMALS]; ps = [RUN2[a][m]["p"] for a in ANIMALS]
    ds = [int(RUN2[a][m]["D"].split("/")[0]) for a in ANIMALS]
    print(f"  {m}: mean score {sum(ss)/len(ss):+.4f}  mean p {sum(ps)/len(ps):.4f}  "
          f"mean D {sum(ds)/len(ds):.1f}/32")
print(f"\ntotal {(time.time()-t_all)/60:.1f} min")


stage-1 pool: 4096 tokens, 2411 blocked by the union of all 14 animal blocklists

STAGE 1 — generic 'an animal' trigger @L20
  score +0.7811 -> +0.9693 in 66s
  '🐁ᅴהחלט𝓭꞊풂🐁ﲢ'

STAGE 2 — per-animal @L30, 100 steps, seed 1
animal      prior | warm score       p      D says         | cold score       p      D says        
--------------------------------------------------------------------------------------------------------
wolf       0.0505 |    +0.9590  0.9066  32/32 wolf         |    +0.9641  0.2819   8/32 dolphin     
cow        0.0000 |    +0.9565  0.0079   0/32 horse        |    +0.9778  0.2463  12/32 elephant    
mouse      0.0000 |    +0.9739  0.0099   0/32 cat          |    +0.9667  0.0063   0/32 dolphin     
elephant   0.3730 |    +0.9639  0.6414  25/32 elephant     |    +0.9703  0.8038  30/32 elephant    
tiger      0.0734 |    +0.9611  0.9716  32/32 tiger        |    +0.9659  0.8111  28/32 tiger       
lion       0.0128 |    +0.9667  0.7422  23/32 lion         |    +0.9621  0

In [27]:
# === 7.3 What actually predicts success? (prior vs warm start vs diet) ===
# Three candidate explanations for the §7 table, tested against each other.
import torch, math

CARNIVORE = {"wolf", "tiger", "lion", "dog", "cat", "fox", "dolphin", "whale", "eagle", "bear"}

rows = []
for a in ANIMALS:
    w, c = RUN2[a]["warm"], RUN2[a]["cold"]
    s, _ = batch_T(WARM[None], SPEC[a])        # where the stage-1 base sits vs THIS animal's target
    rows.append(dict(a=a, prior=RUN2[a]["prior"], wp=w["p"], cp=c["p"],
                     dw=w["p"] - c["p"], base=s.item()))

print(f"{'animal':<9} {'prior':>7} {'warm p':>8} {'cold p':>8} {'warm-cold':>10} "
      f"{'base score':>11}  diet")
print("-" * 70)
for r in sorted(rows, key=lambda r: -r["dw"]):
    print(f"{r['a']:<9} {r['prior']:>7.4f} {r['wp']:>8.4f} {r['cp']:>8.4f} {r['dw']:>+10.3f} "
          f"{r['base']:>11.4f}  {'carnivore' if r['a'] in CARNIVORE else 'herbivore'}")

def corr(x, y):
    n = len(x); mx, my = sum(x)/n, sum(y)/n
    vx = math.sqrt(sum((i-mx)**2 for i in x)); vy = math.sqrt(sum((j-my)**2 for j in y))
    return sum((i-mx)*(j-my) for i, j in zip(x, y)) / (vx*vy) if vx and vy else float("nan")

dw   = [r["dw"] for r in rows]
base = [r["base"] for r in rows]
lpri = [math.log10(max(r["prior"], 1e-6)) for r in rows]
carn = [1.0 if r["a"] in CARNIVORE else 0.0 for r in rows]
best = [max(r["wp"], r["cp"]) for r in rows]

print(f"\n{'='*70}")
print(f"corr( best p          , log10 prior )                       = {corr(best, lpri):+.3f}   <-- the real one")
print(f"corr( warm-minus-cold , is-carnivore )                      = {corr(dw, carn):+.3f}")
print(f"corr( warm-minus-cold , base-trigger score on that animal ) = {corr(dw, base):+.3f}")

c_dw = [r["dw"] for r in rows if r["a"] in CARNIVORE]
h_dw = [r["dw"] for r in rows if r["a"] not in CARNIVORE]
print(f"\nmean(warm-cold): carnivores {sum(c_dw)/len(c_dw):+.3f} (n={len(c_dw)})  "
      f"herbivores {sum(h_dw)/len(h_dw):+.3f} (n={len(h_dw)})")
print(f"carnivores where warm wins: {sum(1 for x in c_dw if x > 0)}/{len(c_dw)}")

hi = [r for r in rows if r["prior"] >= 0.0077]
lo = [r for r in rows if r["prior"] <= 0.0025]
print(f"\nprior >= 0.0077 (n={len(hi)}): best p mean {sum(max(r['wp'],r['cp']) for r in hi)/len(hi):.3f}")
print(f"prior <= 0.0025 (n={len(lo)}): best p mean {sum(max(r['wp'],r['cp']) for r in lo)/len(lo):.3f}")

print("""
READING:
  * The prior dominates (+0.855). Phase-2 logit-GCG had corr = -0.024 and drove crab from a
    0.0000 prior to 0.92 — so the SAE-space objective AMPLIFIES a belief the model already
    holds and cannot INSTALL one from nothing. Sharpest phase-2/phase-3 contrast so far.
  * The warm start is a null result: 9/14 targets answered either way, mean p 0.5385 vs 0.5235.
  * The carnivore split looked exact at 10/14 animals (6/6, p~0.07) and died on the tail:
    carnivores where warm wins 5/10, and the three largest warm LOSSES are all carnivores.
  * Base proximity does not explain the warm start either (-0.283, slightly backwards). Note
    mouse has the highest base score (0.959) because the stage-1 trigger contains the mouse
    emoji twice — the union blocklist covered animal NAMES but not emoji. Fix before re-running.
  * n=1 seed per cell: any |warm-cold| below ~0.15 is not distinguishable from seed noise,
    which covers 8 of these 14 rows.
""")


animal      prior   warm p   cold p  warm-cold  base score  diet
----------------------------------------------------------------------
wolf       0.0505   0.9066   0.2819     +0.625      0.7409  carnivore
whale      0.0077   0.9713   0.4943     +0.477      0.4346  carnivore
dog        0.0113   0.9857   0.8174     +0.168      0.6630  carnivore
tiger      0.0734   0.9716   0.8111     +0.160      0.7701  carnivore
lion       0.0128   0.7422   0.6400     +0.102      0.7278  carnivore
mouse      0.0000   0.0099   0.0063     +0.004      0.9589  herbivore
bear       0.0002   0.0126   0.0333     -0.021      0.7714  carnivore
dolphin    0.3730   0.9424   0.9685     -0.026      0.7346  carnivore
horse      0.0002   0.0306   0.0788     -0.048      0.7726  herbivore
elephant   0.3730   0.6414   0.8038     -0.162      0.6891  herbivore
cow        0.0000   0.0079   0.2463     -0.238      0.8183  herbivore
cat        0.0505   0.6257   0.8665     -0.241      0.7687  carnivore
fox        0.0025   0.01